# 11b — The necessity test (E19) on Alberta: leave-EFG-out anchors (T2) and the conditional no-EFG ensemble (T3)

Mirror of the parent's `18b_e19_solves` (study plan v0.17.2; AB spec v0.5 M16). Runs on the curated-block re-solve
(`VERSION = "v3.1"`, after 10 and 11). **T2** (12 × ~1 s at Alberta scale): for every design formulation, the anchor solved
with every EFG multiplier at 0 → `runs_v3.1/ab_l/A/e19_t2/<formulation_id>/run/portfolio.tif`. Core cells absent from every
no-EFG anchor are EFG-necessary by counterfactual (11c compares them with the adequacy-forced set, T1).
**T3** (~1 h at Alberta scale): the full no-EFG ensemble at the APPLIED band (anchor + 50 MGA members + 50 guarded
members per formulation) — runs ONLY if `spec/v3.1/e19_gate.json`, written by 11c, says the core is predominantly forced
(> 50%). Resumable; live internet (WLS). Kernel `R (y2y)`.

In [ ]:
ANALYSIS <- "ab_y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))
HERE <- file.path(PROJ, "analyses", "alberta_prioritization")
py <- file.path(PROJ, ".venv", "bin", "python")
code <- paste0("import config; print(config.write_manifest(analysis='", ANALYSIS, "', ",
               "handoff_dir=config.AB_HANDOFF_DIR, manifest_path=config.AB_HANDOFF_DIR/'manifest.json'))")
out <- suppressWarnings(system2(py, c("-c", shQuote(code)), stdout = TRUE, stderr = TRUE))
if (!is.null(attr(out, "status")) && attr(out, "status") != 0) stop(paste(out, collapse = "\n"))
mpath <- file.path(PROJ, "input_data", "aligned_stack_ab", "manifest.json")
VERSION <- "v3.1"      # the necessity test runs on the curated-block re-solve only (parent v0.17.2)
stopifnot("the necessity test (E19) runs on the curated-block re-solve only" = VERSION != "v1")
MANIFEST_REL <- sprintf("analyses/alberta_prioritization/spec/manifest_%s.csv", VERSION); FREEZE_REL <- sprintf("analyses/alberta_prioritization/spec/manifest_%s.sha256", VERSION)
REC_REL <- sprintf("analyses/alberta_prioritization/spec/%s", VERSION)
EFG_SUBDIR_EXPECTED <- paste0("iucn_efg_", sub("\\..*$", "", VERSION))
MAN <- read.csv(file.path(PROJ, MANIFEST_REL), stringsAsFactors = FALSE)
dig <- strsplit(readLines(file.path(PROJ, FREEZE_REL))[1], "  ")[[1]][1]
stopifnot(identical(unname(tools::sha256sum(file.path(PROJ, MANIFEST_REL))[[1]]), dig), nrow(MAN) == 12)
LEVEL <- unique(MAN$budget_level); stopifnot(length(LEVEL) == 1)
BUDGET_PCT <- unique(MAN$budget_pct); stopifnot(length(BUDGET_PCT) == 1)
RUNS_REL <- file.path(sprintf("analyses/alberta_prioritization/runs_%s/ab_l", VERSION), LEVEL); RUNS <- file.path(PROJ, RUNS_REL)
REAL245 <- "input_data/aligned_stack_ab/climate_realizations/macrorefugia_245_2071_2100.tif"
SC <- jsonlite::read_json(file.path(HERE, "spec", "scenarios_ab_v1.json")); BLOCKS <- lapply(SC$`_meta`$blocks, unlist)
FLOOR_G <- unique(MAN$floor_g); stopifnot(length(FLOOR_G) == 1)
ctx585 <- pr_setup(mpath, PROJ); ctx585 <- modifyList(ctx585, pr_ingest(ctx585))
ctx245 <- pr_setup(mpath, PROJ)
ctx245$layers$path[ctx245$layers$name == "climate_type_macrorefugia"] <- REAL245
ctx245 <- modifyList(ctx245, pr_ingest(ctx245))
efg_names <- ctx585$layers$name[ctx585$layers$role == "feature_efg"]
stopifnot(length(efg_names) > 0, all(grepl(paste0("/", EFG_SUBDIR_EXPECTED, "/"), ctx585$layers$path[ctx585$layers$role == "feature_efg"])))
base_for <- function(row, results_dir) {
  b <- if (grepl("^ssp245", row$climate_level)) ctx245 else ctx585
  b <- pr_override(b, budget_pct = BUDGET_PCT, results_dir = results_dir, results_subdir = "_base")
  modifyList(b, pr_planning_units(b))
}
form_wt  <- function(row) list(w = jsonlite::fromJSON(row$weight_vector), t = jsonlite::fromJSON(row$target_vector))
no_efg <- function(w) { for (f in efg_names) w[[f]] <- 0; w }        # every EFG multiplier -> 0 (the parent's E17 T3 convention)
run_single <- function(row, w, t, out_rel, artifact = "run") {
  done <- file.path(PROJ, out_rel, artifact, "run_summary.json")
  if (file.exists(done)) { cat(sprintf("   %s exists -- skipped\n", out_rel)); return(invisible(NULL)) }
  actx <- do.call(pr_override, c(list(base_for(row, out_rel), targets = t, feature_weight_multipliers = w,
      results_subdir = artifact, solver = "gurobi", decision_type = "binary", opt_gap = 1e-4, portfolio_n = 1)))
  actx <- modifyList(actx, pr_weights(actx)); actx <- modifyList(actx, pr_targets(actx)); actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  sv <- pr_solve(actx); actx$s <- sv$s; actx$timing <- sv$timing; actx$n_sol <- sv$n_sol; actx$sol_attrs <- sv$sol_attrs
  actx <- modifyList(actx, pr_summaries(actx)); pr_write_outputs(actx); invisible(NULL)
}
cat(sprintf("VERSION %s | level %s | %d design formulations | %d EFG features zeroed for the counterfactuals\n", VERSION, LEVEL, nrow(MAN), length(efg_names)))


In [ ]:
# ---- T2: leave-EFG-out anchors, one per design formulation (~1 s each at Alberta scale) ------------------------
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]; wt <- form_wt(row)
  cat(sprintf("== %s (%d/%d)\n", row$formulation_id, i, nrow(MAN)))
  run_single(row, no_efg(wt$w), wt$t, file.path(RUNS_REL, "e19_t2", row$formulation_id))
}
cat("T2 complete -- next: 11c_ab_e19_analysis (T1 + T2 agreement + the T3 gate)\n")


In [ ]:
# ---- T3 (CONDITIONAL): the no-EFG ensemble at the APPLIED band -- anchors + MGA + guarded members, EFG multipliers 0 -----
gate_f <- file.path(PROJ, REC_REL, "e19_gate.json")
gate <- if (file.exists(gate_f)) jsonlite::read_json(gate_f) else NULL
if (is.null(gate)) {
  cat("T3 gate not written yet -- run 11c_ab_e19_analysis first (it decides whether the core is predominantly forced)\n")
} else if (!isTRUE(gate$t3_triggered)) {
  cat(sprintf("T3 NOT triggered: forced share of the core %.1f%% (rule: > 50%%) -- the no-EFG ensemble is not run\n", 100 * gate$forced_share_core_all))
} else {
  G_APPLIED <- as.numeric(gate$applied_g); TAG <- sprintf("g%02d", round(100 * G_APPLIED))
  cat(sprintf("T3 TRIGGERED: forced share of the core %.1f%% -- solving the no-EFG ensemble at the applied band g = %.0f%% (~1 h at Alberta scale)\n", 100 * gate$forced_share_core_all, 100 * G_APPLIED))
  for (i in seq_len(nrow(MAN))) {
    row <- MAN[i, ]; wt <- form_wt(row); out_rel <- file.path(RUNS_REL, "e19_t3", row$formulation_id); cd <- file.path(PROJ, out_rel)
    dir.create(cd, recursive = TRUE, showWarnings = FALSE)
    cat(sprintf("\n===================== %s (%d/%d) =====================\n", row$formulation_id, i, nrow(MAN)))
    if (file.exists(file.path(cd, sprintf("mga_guard_%s.tif", TAG)))) { cat("   exists -- skipped\n"); next }
    actx <- pr_override(base_for(row, out_rel), targets = wt$t, feature_weight_multipliers = no_efg(wt$w), results_subdir = "mga_build",
                        solver = "gurobi", decision_type = "binary", opt_gap = 1e-4, portfolio_n = 1)
    actx <- modifyList(actx, pr_weights(actx)); actx <- modifyList(actx, pr_targets(actx)); actx <- modifyList(actx, pr_penalty_matrices(actx))
    bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
    cm <- mga_compile(actx); anchor <- mga_anchor(cm, opt_gap = row$opt_gap)
    if (!file.exists(file.path(cd, sprintf("mga_%s.tif", TAG)))) {
      gen <- mga_generate(cm, anchor, g = G_APPLIED, k = row$k_requested); mga_write(gen, cm, actx$cost, cd, TAG)
    }
    r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r)); v[cm$pu_index] <- as.integer(anchor$x); terra::values(r) <- v
    terra::writeRaster(r, file.path(cd, "anchor.tif"), overwrite = TRUE, datatype = "INT1U", NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
    jsonlite::write_json(list(formulation_id = row$formulation_id, experiment = "E19 T3 no-EFG ensemble (Alberta)", anchor_objective = anchor$z,
                              anchor_gap = anchor$gap, anchor_runtime_s = anchor$runtime, weight_vector = no_efg(wt$w), target_vector = wt$t,
                              k = row$k_requested, g = G_APPLIED, level = LEVEL, created_utc = format(Sys.time(), tz = "UTC")),
                         file.path(cd, "formulation_meta.json"), auto_unbox = TRUE, pretty = TRUE, digits = 10)
    gg <- mga_generate(cm, anchor, g = G_APPLIED, k = row$k_requested, floors = list(ctx = actx, blocks = BLOCKS, g = FLOOR_G))
    mga_write(gg, cm, actx$cost, cd, sprintf("guard_%s", TAG))
    cat(sprintf("   wrote anchor, MGA members, guarded members for %s\n", row$formulation_id))
  }
  cat("T3 complete -- re-run 11c_ab_e19_analysis for the F_noEFG surface\n")
}
